In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

# Haversine distance formula to calculate real-world kilometers between lat/lon points
def haversine_dist(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    dLat = np.radians(lat2 - lat1)
    dLon = np.radians(lon2 - lon1)
    a = np.sin(dLat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def get_pairwise_distances(centroids_df):
    lats = centroids_df['latitude'].values
    lons = centroids_df['longitude'].values
    distances = []
    
    # Calculate distance between every unique pair of centroids
    for i, j in combinations(range(len(centroids_df)), 2):
        distances.append(haversine_dist(lats[i], lons[i], lats[j], lons[j]))
    return np.array(distances)

def analyze_clusters(k):
    # Load the generated files
    clusters_df = pd.read_csv('geolocation-prediction/clustering_data/all_image_clusters.csv')
    centroids_df = pd.read_csv(f'geolocation-prediction/clustering_data/centroids_{k}.csv')
    
    # Isolate only the original 19K training images
    orig_df = clusters_df[clusters_df['dataset_source'] == 'original']
    
    # 1. Analyze Image Counts
    cluster_col = f'cluster_{k}'
    image_counts = orig_df[cluster_col].value_counts().values
    
    print(f"=== ANALYSIS FOR K={k} (Original Dataset Only) ===")
    print("Images per Cluster:")
    print(f"  Minimum images: {np.min(image_counts)}")
    print(f"  Maximum images: {np.max(image_counts)}")
    print(f"  Mean images:    {np.mean(image_counts):.1f}")
    print(f"  Median images:  {np.median(image_counts):.1f}")
    
    # 2. Analyze Centroid Distances
    distances = get_pairwise_distances(centroids_df)
    
    print(f"\nPairwise Distances Between Centroids:")
    print(f"  Minimum distance: {np.min(distances):,.1f} km")
    print(f"  Maximum distance: {np.max(distances):,.1f} km")
    print(f"  Mean distance:    {np.mean(distances):,.1f} km")
    print(f"  Median distance:  {np.median(distances):,.1f} km")
    print("-" * 50 + "\n")

# Run the analysis for both heads
analyze_clusters(16)
analyze_clusters(160)

C:\Users\Yash T\AppData\Local\Temp\ipykernel_21196\2943690725.py:26: DtypeWarning: Columns (0: image_id) have mixed types. Specify dtype option on import or set low_memory=False.
  clusters_df = pd.read_csv('geolocation-prediction/clustering_data/all_image_clusters.csv')


=== ANALYSIS FOR K=16 (Original Dataset Only) ===
Images per Cluster:
  Minimum images: 564
  Maximum images: 1925
  Mean images:    1187.6
  Median images:  1219.0

Pairwise Distances Between Centroids:
  Minimum distance: 2,001.1 km
  Maximum distance: 18,797.9 km
  Mean distance:    9,507.4 km
  Median distance:  9,346.3 km
--------------------------------------------------

=== ANALYSIS FOR K=160 (Original Dataset Only) ===
Images per Cluster:
  Minimum images: 11
  Maximum images: 289
  Mean images:    118.8
  Median images:  123.0

Pairwise Distances Between Centroids:
  Minimum distance: 398.4 km
  Maximum distance: 19,811.3 km
  Mean distance:    9,366.9 km
  Median distance:  9,401.6 km
--------------------------------------------------



C:\Users\Yash T\AppData\Local\Temp\ipykernel_21196\2943690725.py:26: DtypeWarning: Columns (0: image_id) have mixed types. Specify dtype option on import or set low_memory=False.
  clusters_df = pd.read_csv('geolocation-prediction/clustering_data/all_image_clusters.csv')


In [2]:
import pandas as pd
import numpy as np

# Standalone NumPy Haversine calculation
def haversine_numpy(lat1, lon1, lats2, lons2):
    R = 6371.0  # Earth radius in kilometers
    
    # Convert all to radians
    lat1, lon1, lats2, lons2 = map(np.radians, [lat1, lon1, lats2, lons2])
    
    dlat = lats2 - lat1
    dlon = lons2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lats2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def print_top_5_nearest_clusters(centroids_csv, target_cluster_idx=0):
    df = pd.read_csv(centroids_csv)
    
    # Grab the target cluster's coordinates
    target_row = df.iloc[target_cluster_idx]
    target_lat = target_row['latitude']
    target_lon = target_row['longitude']
    target_name = target_row['cluster_id']
    
    # Calculate distances from target to ALL clusters simultaneously
    distances = haversine_numpy(target_lat, target_lon, df['latitude'].values, df['longitude'].values)
    
    # Add the distances directly to the dataframe for easy sorting
    df['distance_km'] = distances
    
    # Sort the dataframe by distance. 
    # .iloc[1:6] grabs rows 1 through 5, skipping row 0 (which is the target itself at 0km)
    nearest_df = df.sort_values(by='distance_km').iloc[1:6]
    
    print(f"Top 5 closest clusters to {target_name} (Index {target_cluster_idx}):")
    for i, (_, row) in enumerate(nearest_df.iterrows(), 1):
        print(f"  {i}. {row['cluster_id']} -> {row['distance_km']:,.1f} km away")

# ==========================================
# Run for Head 1 (16 Clusters)
# ==========================================
print("=== HEAD 1 (16 Clusters) ===")
print_top_5_nearest_clusters('geolocation-prediction/clustering_data/centroids_16.csv', target_cluster_idx=0)
print("\n")

# ==========================================
# Run for Head 2 (160 Clusters)
# ==========================================
print("=== HEAD 2 (160 Clusters) ===")
print_top_5_nearest_clusters('geolocation-prediction/clustering_data/centroids_160.csv', target_cluster_idx=0)

=== HEAD 1 (16 Clusters) ===
Top 5 closest clusters to CLUSTER_0 (Index 0):
  1. CLUSTER_11 -> 3,886.6 km away
  2. CLUSTER_10 -> 4,314.0 km away
  3. CLUSTER_6 -> 4,532.8 km away
  4. CLUSTER_14 -> 5,177.9 km away
  5. CLUSTER_4 -> 5,744.8 km away


=== HEAD 2 (160 Clusters) ===
Top 5 closest clusters to CLUSTER_0 (Index 0):
  1. CLUSTER_30 -> 621.0 km away
  2. CLUSTER_92 -> 674.4 km away
  3. CLUSTER_67 -> 834.5 km away
  4. CLUSTER_121 -> 883.1 km away
  5. CLUSTER_133 -> 996.6 km away


In [3]:
import torch

print("Loading embeddings...")
data = torch.load("geolocation-prediction/extracted_features_dinov2_large.pt")

features = data['features']
latitudes = data['latitudes']
longitudes = data['longitudes']
image_ids = data['image_ids']

print("Splitting datasets...")
# Create a boolean mask: True if the ID starts with 'IMG_', False otherwise
is_original = torch.tensor([str(img_id).startswith('IMG_') for img_id in image_ids])

# Apply the mask to get the Original 19K dataset
orig_features = features[is_original]
orig_latitudes = latitudes[is_original]
orig_longitudes = longitudes[is_original]
# (Optional) keep original IDs if needed
orig_ids = [img_id for img_id in image_ids if str(img_id).startswith('IMG_')]

# Invert the mask (~) to get the Extra 210K dataset
extra_features = features[~is_original]
extra_latitudes = latitudes[~is_original]
extra_longitudes = longitudes[~is_original]

print(f"Original Dataset Size: {len(orig_features)}")
print(f"Extra Dataset Size: {len(extra_features)}")

Loading embeddings...
Splitting datasets...
Original Dataset Size: 19002
Extra Dataset Size: 210122


In [4]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# --- 1. Load Targets & Align with Features ---
print("Aligning targets...")
# Load the clustering mapping we created earlier
df_clusters = pd.read_csv("geolocation-prediction/clustering_data/all_image_clusters.csv")

# Create a fast dictionary mapping: image_id (string) -> (cluster_16_int, cluster_160_int)
id_to_targets = {}
for _, row in df_clusters.iterrows():
    # Extract just the integer from "CLUSTER_X"
    c16 = int(row['cluster_16'].split('_')[1])
    c160 = int(row['cluster_160'].split('_')[1])
    id_to_targets[str(row['image_id'])] = (c16, c160)

# Build target tensors in the exact same order as your `image_ids`
targets_16, targets_160 = [], []
for img_id in image_ids:
    c16, c160 = id_to_targets[str(img_id)]
    targets_16.append(c16)
    targets_160.append(c160)
targets_16 = torch.tensor(targets_16, dtype=torch.long)
targets_160 = torch.tensor(targets_160, dtype=torch.long)

# Apply your existing boolean mask to slice the targets
orig_targets_16 = targets_16[is_original]
orig_targets_160 = targets_160[is_original]
extra_targets_16 = targets_16[~is_original]
extra_targets_160 = targets_160[~is_original]

Aligning targets...


C:\Users\Yash T\AppData\Local\Temp\ipykernel_21196\1216991751.py:9: DtypeWarning: Columns (0: image_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clusters = pd.read_csv("geolocation-prediction/clustering_data/all_image_clusters.csv")


In [5]:
# --- 2. Model Definition ---
class GeolocationMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(1024, 640),
            nn.BatchNorm1d(640),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(640, 320),
            nn.BatchNorm1d(320),
            nn.ReLU()
        )
        self.head_16 = nn.Linear(320, 16)
        self.head_160 = nn.Linear(320, 160)
        
    def forward(self, x):
        features = self.shared(x)
        out_16 = self.head_16(features)
        out_160 = self.head_160(features)
        return out_16, out_160

In [6]:
# --- 3. Fast In-Memory Dataloaders ---
batch_size = 2048

# Phase 1 uses all data. We concat original and extra.
all_features = torch.cat([orig_features, extra_features], dim=0)
all_targets_16 = torch.cat([orig_targets_16, extra_targets_16], dim=0)
all_targets_160 = torch.cat([orig_targets_160, extra_targets_160], dim=0)
dataset_all = TensorDataset(all_features, all_targets_16, all_targets_160)
loader_all = DataLoader(dataset_all, batch_size=batch_size, shuffle=True, num_workers=0)
dataset_orig = TensorDataset(orig_features, orig_targets_16, orig_targets_160)
loader_orig = DataLoader(dataset_orig, batch_size=batch_size, shuffle=True, num_workers=0)

In [7]:
# --- 4. Haversine Soft-Target Loss ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

soft_targets_16 = torch.load("geolocation-prediction/clustering_data/loss_matrix_16.pt").to(device)
soft_targets_160 = torch.load("geolocation-prediction/clustering_data/loss_matrix_160.pt").to(device)

def haversine_cross_entropy(logits, true_indices, soft_target_matrix):
    # Retrieve the soft probability distributions for the ground truth clusters
    batch_targets = soft_target_matrix[true_indices]
    # Standard PyTorch cross entropy calculates loss against the soft distributions
    return nn.functional.cross_entropy(logits, batch_targets)

In [8]:
# --- 5. Training Loop Setup ---
model = GeolocationMLP().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

def train_phase(phase_name, dataloader, epochs):
    print(f"\n=== Starting {phase_name} ===")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        
        for batch_features, batch_t16, batch_t160 in dataloader:
            batch_features = batch_features.to(device)
            batch_t16 = batch_t16.to(device)
            batch_t160 = batch_t160.to(device)
            
            optimizer.zero_grad()
            
            out_16, out_160 = model(batch_features)
            
            loss_16 = haversine_cross_entropy(out_16, batch_t16, soft_targets_16)
            loss_160 = haversine_cross_entropy(out_160, batch_t160, soft_targets_160)
            
            # Weighted Loss
            loss = (0.1 * loss_16) + (0.9 * loss_160)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        avg_loss = total_loss / len(dataloader)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")
        
        # Step the scheduler based on training loss
        scheduler.step(avg_loss)

In [7]:
# --- 6. Execution ---
# Phase 1: Train on full dataset
train_phase("Phase 1: Full Dataset (80 Epochs)", loader_all, epochs=80)


=== Starting Phase 1: Full Dataset (80 Epochs) ===
Epoch [01/80] - Loss: 2.1507 | LR: 0.001000
Epoch [02/80] - Loss: 1.1775 | LR: 0.001000
Epoch [03/80] - Loss: 0.9419 | LR: 0.001000
Epoch [04/80] - Loss: 0.8037 | LR: 0.001000
Epoch [05/80] - Loss: 0.7071 | LR: 0.001000
Epoch [06/80] - Loss: 0.6302 | LR: 0.001000
Epoch [07/80] - Loss: 0.5680 | LR: 0.001000
Epoch [08/80] - Loss: 0.5189 | LR: 0.001000
Epoch [09/80] - Loss: 0.4753 | LR: 0.001000
Epoch [10/80] - Loss: 0.4375 | LR: 0.001000
Epoch [11/80] - Loss: 0.4077 | LR: 0.001000
Epoch [12/80] - Loss: 0.3799 | LR: 0.001000
Epoch [13/80] - Loss: 0.3576 | LR: 0.001000
Epoch [14/80] - Loss: 0.3391 | LR: 0.001000
Epoch [15/80] - Loss: 0.3199 | LR: 0.001000
Epoch [16/80] - Loss: 0.3065 | LR: 0.001000
Epoch [17/80] - Loss: 0.2939 | LR: 0.001000
Epoch [18/80] - Loss: 0.2839 | LR: 0.001000
Epoch [19/80] - Loss: 0.2747 | LR: 0.001000
Epoch [20/80] - Loss: 0.2639 | LR: 0.001000
Epoch [21/80] - Loss: 0.2572 | LR: 0.001000
Epoch [22/80] - Loss: 0.

In [8]:
# Reset scheduler patience so it behaves freshly for phase 2
optimizer = optim.AdamW(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# Phase 2: Fine-tune strictly on original dataset
train_phase("Phase 2: Original Dataset Only (20 Epochs)", loader_orig, epochs=20)


=== Starting Phase 2: Original Dataset Only (20 Epochs) ===
Epoch [01/20] - Loss: 0.3865 | LR: 0.000200
Epoch [02/20] - Loss: 0.2980 | LR: 0.000200
Epoch [03/20] - Loss: 0.2707 | LR: 0.000200
Epoch [04/20] - Loss: 0.2569 | LR: 0.000200
Epoch [05/20] - Loss: 0.2434 | LR: 0.000200
Epoch [06/20] - Loss: 0.2369 | LR: 0.000200
Epoch [07/20] - Loss: 0.2327 | LR: 0.000200
Epoch [08/20] - Loss: 0.2261 | LR: 0.000200
Epoch [09/20] - Loss: 0.2219 | LR: 0.000200
Epoch [10/20] - Loss: 0.2179 | LR: 0.000200
Epoch [11/20] - Loss: 0.2173 | LR: 0.000200
Epoch [12/20] - Loss: 0.2145 | LR: 0.000200
Epoch [13/20] - Loss: 0.2119 | LR: 0.000200
Epoch [14/20] - Loss: 0.2078 | LR: 0.000200
Epoch [15/20] - Loss: 0.2059 | LR: 0.000200
Epoch [16/20] - Loss: 0.2060 | LR: 0.000200
Epoch [17/20] - Loss: 0.2056 | LR: 0.000200
Epoch [18/20] - Loss: 0.2044 | LR: 0.000200
Epoch [19/20] - Loss: 0.2011 | LR: 0.000200
Epoch [20/20] - Loss: 0.1992 | LR: 0.000200


In [9]:
# Save the final model weights
torch.save(model.state_dict(), "geolocation-prediction/saved_models/layer1_model_weights.pth")
print("\nSuccess! Model saved to geolocation-prediction/layer1_model_weights.pth")


Success! Model saved to geolocation-prediction/layer1_model_weights.pth


In [9]:
# --- 1. Load Test Data & Centroids ---
import os
import numpy as np

SUBMISSION_DIR = "geolocation-prediction/submissions"
os.makedirs(SUBMISSION_DIR, exist_ok=True)
print("Loading data...")
test_data = torch.load("geolocation-prediction/test_extracted_features_dinov2_large.pt")
test_features = test_data['features']
test_image_ids = test_data['image_ids']
df_16 = pd.read_csv("geolocation-prediction/clustering_data/centroids_16.csv")
df_160 = pd.read_csv("geolocation-prediction/clustering_data/centroids_160.csv")
cent_16_map = {int(row['cluster_id'].split('_')[1]): (row['latitude'], row['longitude']) for _, row in df_16.iterrows()}
cent_160_map = {int(row['cluster_id'].split('_')[1]): (row['latitude'], row['longitude']) for _, row in df_160.iterrows()}

Loading data...


In [10]:
# --- 2. Calculate Radii ---
# Haversine distance formula to calculate real-world kilometers between lat/lon points
def haversine_dist(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    dLat = np.radians(lat2 - lat1)
    dLon = np.radians(lon2 - lon1)
    a = np.sin(dLat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

print("Calculating radii bounds...")
radius_map_16 = {}
for i in range(16):
    lat_i, lon_i = cent_16_map[i]
    min_dist = float('inf')
    for j in range(16):
        if i == j: continue
        lat_j, lon_j = cent_16_map[j]
        dist = haversine_dist(lat_i, lon_i, lat_j, lon_j) # Using function from your cluster analysis cell
        if dist < min_dist:
            min_dist = dist
    radius_map_16[i] = min_dist

Calculating radii bounds...


In [11]:
# --- 3. Load Model Weights & Run Inference ---
print("Running inference...")
model.load_state_dict(torch.load("geolocation-prediction/saved_models/layer1_model_weights.pth"))
model.eval()
pred_lats, pred_lons, pred_radii = [], [], []
batch_size = 512
with torch.no_grad():
    for i in range(0, len(test_features), batch_size):
        batch = test_features[i:i+batch_size].to(device)
        
        out_16, out_160 = model(batch)
        
        preds_16 = torch.argmax(out_16, dim=1).cpu().numpy()
        preds_160 = torch.argmax(out_160, dim=1).cpu().numpy()
        
        for p16, p160 in zip(preds_16, preds_160):
            lat, lon = cent_160_map[p160]
            pred_lats.append(lat)
            pred_lons.append(lon)
            pred_radii.append(radius_map_16[p16])

Running inference...


In [12]:
# --- 4. Format & Save Submission ---
submission_df = pd.DataFrame({
    'image_id': test_image_ids,
    'pred_lat': pred_lats,
    'pred_lon': pred_lons,
    'pred_radius_km': pred_radii
})
# Append .jpg for Kaggle format if missing
submission_df['image_id'] = submission_df['image_id'].apply(lambda x: x + ".jpg" if not str(x).endswith('.jpg') else x)
submission_path = os.path.join(SUBMISSION_DIR, "submission_layer1_only.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Done! Saved submission to {submission_path}")

Done! Saved submission to geolocation-prediction/submissions\submission_layer1_only.csv


In [13]:
# --- 1. Helper for 3D Conversion ---
def latlon_to_3d(lats, lons):
    lat_rad = torch.deg2rad(lats)
    lon_rad = torch.deg2rad(lons)
    x = torch.cos(lat_rad) * torch.cos(lon_rad)
    y = torch.cos(lat_rad) * torch.sin(lon_rad)
    z = torch.sin(lat_rad)
    return torch.stack([x, y, z], dim=1)

In [14]:
# --- 2. Calculate Ground Truth Offsets ---
print("Preparing 3D Target Offsets...")
# Convert true lat/lon to 3D Cartesian
orig_true_3d = latlon_to_3d(orig_latitudes, orig_longitudes)
extra_true_3d = latlon_to_3d(extra_latitudes, extra_longitudes)

# Load the 160 Centroids and ensure they are sorted by index
df_160 = pd.read_csv("geolocation-prediction/clustering_data/centroids_160.csv")
df_160['idx'] = df_160['cluster_id'].apply(lambda c: int(c.split('_')[1]))
df_160 = df_160.sort_values('idx')
centroids_3d_160 = torch.tensor(df_160[['x', 'y', 'z']].values, dtype=torch.float32)

# The Target Offset is the True 3D coordinate minus the Centroid 3D coordinate
orig_target_offsets = orig_true_3d - centroids_3d_160[orig_targets_160]
extra_target_offsets = extra_true_3d - centroids_3d_160[extra_targets_160]

Preparing 3D Target Offsets...


In [15]:
# --- 3. Build Teacher-Forced 1200d Inputs ---
print("Building 1200-dimensional Teacher-Forced Inputs...")
# Move soft targets to CPU temporarily to build the dataset in RAM
st_16_cpu = soft_targets_16.cpu()
st_160_cpu = soft_targets_160.cpu()

# Concatenate: [1024 DINO] + [16 Smoothed Probs] + [160 Smoothed Probs]
orig_X = torch.cat([orig_features, st_16_cpu[orig_targets_16], st_160_cpu[orig_targets_160]], dim=1)
extra_X = torch.cat([extra_features, st_16_cpu[extra_targets_16], st_160_cpu[extra_targets_160]], dim=1)

Building 1200-dimensional Teacher-Forced Inputs...


In [16]:
# --- 4. Fast In-Memory Dataloaders ---
all_X = torch.cat([orig_X, extra_X], dim=0)
all_offsets = torch.cat([orig_target_offsets, extra_target_offsets], dim=0)
all_targets_160 = torch.cat([orig_targets_160, extra_targets_160], dim=0)

dataset_l2_all = TensorDataset(all_X, all_offsets, all_targets_160)
loader_l2_all = DataLoader(dataset_l2_all, batch_size=batch_size, shuffle=True, num_workers=0)

dataset_l2_orig = TensorDataset(orig_X, orig_target_offsets, orig_targets_160)
loader_l2_orig = DataLoader(dataset_l2_orig, batch_size=batch_size, shuffle=True, num_workers=0)

In [17]:
# --- 5. Multi-Head Layer 2 Architecture ---
class Layer2OffsetMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1200, 640),
            nn.BatchNorm1d(640),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(640, 320),
            nn.BatchNorm1d(320),
            nn.ReLU(),
            nn.Linear(320, 480)  # 160 clusters * 3 coordinates
        )
        
    def forward(self, x):
        out = self.net(x)
        # Reshape to (Batch, 160 clusters, 3 coordinates)
        return out.view(-1, 160, 3)

In [18]:
# --- 6. Training Setup ---
model_l2 = Layer2OffsetMLP().to(device)
optimizer_l2 = optim.AdamW(model_l2.parameters(), lr=1e-3)
scheduler_l2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_l2, mode='min', factor=0.5, patience=1)
criterion = nn.MSELoss()

def train_l2_phase(phase_name, dataloader, epochs):
    print(f"\n=== Starting {phase_name} ===")
    for epoch in range(epochs):
        model_l2.train()
        total_loss = 0.0
        
        for batch_X, batch_offset, batch_t160 in dataloader:
            batch_X = batch_X.to(device)
            batch_offset = batch_offset.to(device)
            batch_t160 = batch_t160.to(device)
            
            optimizer_l2.zero_grad()
            
            # Forward pass: Predicts offsets for ALL 160 clusters
            all_preds = model_l2(batch_X)
            
            # Slice out ONLY the 3D offset for the true cluster to compute loss against
            batch_indices = torch.arange(batch_X.size(0))
            specific_preds = all_preds[batch_indices, batch_t160, :]
            
            loss = criterion(specific_preds, batch_offset)
            
            loss.backward()
            optimizer_l2.step()
            total_loss += loss.item()
            
        avg_loss = total_loss / len(dataloader)
        current_lr = optimizer_l2.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:02d}/{epochs}] - MSE Loss: {avg_loss:.6f} | LR: {current_lr:.6f}")
        
        scheduler_l2.step(avg_loss)

In [15]:
# --- 7. Execution ---
train_l2_phase("Layer 2 Phase 1: Full Dataset (80 Epochs)", loader_l2_all, epochs=80)


=== Starting Layer 2 Phase 1: Full Dataset (80 Epochs) ===
Epoch [01/80] - MSE Loss: 0.015718 | LR: 0.001000
Epoch [02/80] - MSE Loss: 0.002294 | LR: 0.001000
Epoch [03/80] - MSE Loss: 0.001668 | LR: 0.001000
Epoch [04/80] - MSE Loss: 0.001357 | LR: 0.001000
Epoch [05/80] - MSE Loss: 0.001172 | LR: 0.001000
Epoch [06/80] - MSE Loss: 0.001040 | LR: 0.001000
Epoch [07/80] - MSE Loss: 0.000947 | LR: 0.001000
Epoch [08/80] - MSE Loss: 0.000874 | LR: 0.001000
Epoch [09/80] - MSE Loss: 0.000798 | LR: 0.001000
Epoch [10/80] - MSE Loss: 0.000732 | LR: 0.001000
Epoch [11/80] - MSE Loss: 0.000677 | LR: 0.001000
Epoch [12/80] - MSE Loss: 0.000644 | LR: 0.001000
Epoch [13/80] - MSE Loss: 0.000600 | LR: 0.001000
Epoch [14/80] - MSE Loss: 0.000567 | LR: 0.001000
Epoch [15/80] - MSE Loss: 0.000533 | LR: 0.001000
Epoch [16/80] - MSE Loss: 0.000500 | LR: 0.001000
Epoch [17/80] - MSE Loss: 0.000485 | LR: 0.001000
Epoch [18/80] - MSE Loss: 0.000455 | LR: 0.001000
Epoch [19/80] - MSE Loss: 0.000437 | LR:

In [16]:
optimizer_l2 = optim.AdamW(model_l2.parameters(), lr=2e-4)
scheduler_l2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_l2, mode='min', factor=0.5, patience=1)
train_l2_phase("Layer 2 Phase 2: Original Dataset Only (20 Epochs)", loader_l2_orig, epochs=20)


=== Starting Layer 2 Phase 2: Original Dataset Only (20 Epochs) ===
Epoch [01/20] - MSE Loss: 0.005665 | LR: 0.000200
Epoch [02/20] - MSE Loss: 0.001528 | LR: 0.000200
Epoch [03/20] - MSE Loss: 0.000881 | LR: 0.000200
Epoch [04/20] - MSE Loss: 0.000663 | LR: 0.000200
Epoch [05/20] - MSE Loss: 0.000551 | LR: 0.000200
Epoch [06/20] - MSE Loss: 0.000472 | LR: 0.000200
Epoch [07/20] - MSE Loss: 0.000428 | LR: 0.000200
Epoch [08/20] - MSE Loss: 0.000386 | LR: 0.000200
Epoch [09/20] - MSE Loss: 0.000356 | LR: 0.000200
Epoch [10/20] - MSE Loss: 0.000328 | LR: 0.000200
Epoch [11/20] - MSE Loss: 0.000308 | LR: 0.000200
Epoch [12/20] - MSE Loss: 0.000288 | LR: 0.000200
Epoch [13/20] - MSE Loss: 0.000271 | LR: 0.000200
Epoch [14/20] - MSE Loss: 0.000255 | LR: 0.000200
Epoch [15/20] - MSE Loss: 0.000242 | LR: 0.000200
Epoch [16/20] - MSE Loss: 0.000226 | LR: 0.000200
Epoch [17/20] - MSE Loss: 0.000218 | LR: 0.000200
Epoch [18/20] - MSE Loss: 0.000209 | LR: 0.000200
Epoch [19/20] - MSE Loss: 0.000

In [17]:
# Safely save to a completely separate file!
torch.save(model_l2.state_dict(), "geolocation-prediction/saved_models/layer2_model_weights.pth")
print("\nSuccess! Layer 2 Model saved to geolocation-prediction/saved_models/layer2_model_weights.pth")


Success! Layer 2 Model saved to geolocation-prediction/saved_models/layer2_model_weights.pth


In [19]:
# --- 1. Load Both Models from the saved_models Folder ---
print("Loading Layer 1 and Layer 2 models...")
model_l1 = GeolocationMLP().to(device)
model_l1.load_state_dict(torch.load("geolocation-prediction/saved_models/layer1_model_weights_v2.pth"))
model_l1.eval()

model_l2 = Layer2OffsetMLP().to(device)
model_l2.load_state_dict(torch.load("geolocation-prediction/saved_models/layer2_model_weights_v2.pth"))
model_l2.eval()

# Move the 160-cluster 3D centroids to the GPU for incredibly fast batched addition
centroids_3d_160_dev = centroids_3d_160.to(device)

Loading Layer 1 and Layer 2 models...


In [20]:
# --- 2. Helper for 3D -> Lat/Lon Conversion ---
def cartesian_to_latlon_tensor(x, y, z):
    # PyTorch math to convert the final 3D offsets back to Earth coordinates
    lon_rad = torch.atan2(y, x)
    hyp = torch.sqrt(x * x + y * y)
    lat_rad = torch.atan2(z, hyp)
    lat = torch.rad2deg(lat_rad)
    lon = torch.rad2deg(lon_rad)
    return lat.cpu().numpy(), lon.cpu().numpy()

In [21]:
# --- 3. Run Cascading Inference ---
print("Running final 2-Layer inference...")
pred_lats, pred_lons, pred_radii = [], [], []
batch_size = 512

with torch.no_grad():
    for i in range(0, len(test_features), batch_size):
        batch_feat = test_features[i:i+batch_size].to(device)
        
        # --- LAYER 1: Classification ---
        out_16_logits, out_160_logits = model_l1(batch_feat)
        
        # Convert logits to probabilities to perfectly simulate the smoothed Teacher Forcing inputs!
        prob_16 = torch.softmax(out_16_logits, dim=1)
        prob_160 = torch.softmax(out_160_logits, dim=1)
        
        # Extract the hard predictions (argmax)
        preds_16 = torch.argmax(out_16_logits, dim=1)
        preds_160 = torch.argmax(out_160_logits, dim=1)
        
        # --- LAYER 2: Regression ---
        # Build the 1200-dimensional input tensor
        l2_inputs = torch.cat([batch_feat, prob_16, prob_160], dim=1)
        
        # Predict all 160 offsets
        all_offsets = model_l2(l2_inputs) # Shape: (Batch, 160, 3)
        
        # Slice out ONLY the offset for the cluster that Layer 1 selected
        batch_indices = torch.arange(batch_feat.size(0))
        specific_offsets = all_offsets[batch_indices, preds_160, :]
        
        # Apply the offset to the base 3D centroid
        base_3d = centroids_3d_160_dev[preds_160]
        final_3d = base_3d + specific_offsets
        
        # --- Format Output ---
        # Convert final 3D point back to Lat/Lon
        lats, lons = cartesian_to_latlon_tensor(final_3d[:, 0], final_3d[:, 1], final_3d[:, 2])
        
        pred_lats.extend(lats)
        pred_lons.extend(lons)
        
        # Radius calculation remains identical (using the macro Head 1 prediction)
        for p16 in preds_16.cpu().numpy():
            pred_radii.append(radius_map_16[p16] * 0.2)

Running final 2-Layer inference...


In [22]:
# --- 4. Save Final Submission ---
print("Formatting Kaggle CSV...")
submission_df = pd.DataFrame({
    'image_id': test_image_ids,
    'pred_lat': pred_lats,
    'pred_lon': pred_lons,
    'pred_radius_km': pred_radii
})

# Append .jpg for Kaggle format if it's missing
submission_df['image_id'] = submission_df['image_id'].apply(lambda x: x + ".jpg" if not str(x).endswith('.jpg') else x)
final_submission_path = "geolocation-prediction/submissions/submission_final_layer1_and_2_v2.csv"
submission_df.to_csv(final_submission_path, index=False)
print(f"Done! Final submission saved to {final_submission_path}")

Formatting Kaggle CSV...
Done! Final submission saved to geolocation-prediction/submissions/submission_final_layer1_and_2_v2.csv


In [23]:
# --- 1. Load Pre-trained Weights ---
print("Loading existing models from saved_models...")
model.load_state_dict(torch.load("geolocation-prediction/saved_models/layer1_model_weights.pth"))
model_l2.load_state_dict(torch.load("geolocation-prediction/saved_models/layer2_model_weights.pth"))

Loading existing models from saved_models...


<All keys matched successfully>

In [24]:
# --- 2. Override Optimizers & Schedulers (Resetting LR to 1e-4) ---
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

optimizer_l2 = optim.AdamW(model_l2.parameters(), lr=1e-4)
scheduler_l2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_l2, mode='min', factor=0.5, patience=1)

In [23]:
# --- 3. Fine-Tune Layer 1 ---
train_phase("Layer 1 Fine-Tuning: Full Dataset (20 Epochs)", loader_all, epochs=20)

# Reset patience for the 19K run
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
train_phase("Layer 1 Fine-Tuning: Original Dataset (5 Epochs)", loader_orig, epochs=5)

# Save to a new file so you don't overwrite your 90-score weights!
torch.save(model.state_dict(), "geolocation-prediction/saved_models/layer1_model_weights_v2.pth")


=== Starting Layer 1 Fine-Tuning: Full Dataset (20 Epochs) ===
Epoch [01/20] - Loss: 0.1511 | LR: 0.000100
Epoch [02/20] - Loss: 0.1456 | LR: 0.000100
Epoch [03/20] - Loss: 0.1441 | LR: 0.000100
Epoch [04/20] - Loss: 0.1432 | LR: 0.000100
Epoch [05/20] - Loss: 0.1430 | LR: 0.000100
Epoch [06/20] - Loss: 0.1426 | LR: 0.000100
Epoch [07/20] - Loss: 0.1423 | LR: 0.000100
Epoch [08/20] - Loss: 0.1422 | LR: 0.000100
Epoch [09/20] - Loss: 0.1417 | LR: 0.000100
Epoch [10/20] - Loss: 0.1417 | LR: 0.000100
Epoch [11/20] - Loss: 0.1414 | LR: 0.000100
Epoch [12/20] - Loss: 0.1414 | LR: 0.000100
Epoch [13/20] - Loss: 0.1413 | LR: 0.000100
Epoch [14/20] - Loss: 0.1409 | LR: 0.000100
Epoch [15/20] - Loss: 0.1408 | LR: 0.000100
Epoch [16/20] - Loss: 0.1410 | LR: 0.000100
Epoch [17/20] - Loss: 0.1408 | LR: 0.000100
Epoch [18/20] - Loss: 0.1408 | LR: 0.000100
Epoch [19/20] - Loss: 0.1404 | LR: 0.000100
Epoch [20/20] - Loss: 0.1406 | LR: 0.000100

=== Starting Layer 1 Fine-Tuning: Original Dataset (5 E

In [24]:
# --- 4. Fine-Tune Layer 2 ---
train_l2_phase("Layer 2 Fine-Tuning: Full Dataset (20 Epochs)", loader_l2_all, epochs=20)

# Reset patience for the 19K run
optimizer_l2 = optim.AdamW(model_l2.parameters(), lr=2e-5)
scheduler_l2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_l2, mode='min', factor=0.5, patience=1)
train_l2_phase("Layer 2 Fine-Tuning: Original Dataset (5 Epochs)", loader_l2_orig, epochs=5)


=== Starting Layer 2 Fine-Tuning: Full Dataset (20 Epochs) ===
Epoch [01/20] - MSE Loss: 0.000096 | LR: 0.000100
Epoch [02/20] - MSE Loss: 0.000060 | LR: 0.000100
Epoch [03/20] - MSE Loss: 0.000054 | LR: 0.000100
Epoch [04/20] - MSE Loss: 0.000053 | LR: 0.000100
Epoch [05/20] - MSE Loss: 0.000051 | LR: 0.000100
Epoch [06/20] - MSE Loss: 0.000050 | LR: 0.000100
Epoch [07/20] - MSE Loss: 0.000050 | LR: 0.000100
Epoch [08/20] - MSE Loss: 0.000049 | LR: 0.000100
Epoch [09/20] - MSE Loss: 0.000049 | LR: 0.000100
Epoch [10/20] - MSE Loss: 0.000049 | LR: 0.000100
Epoch [11/20] - MSE Loss: 0.000048 | LR: 0.000100
Epoch [12/20] - MSE Loss: 0.000048 | LR: 0.000100
Epoch [13/20] - MSE Loss: 0.000047 | LR: 0.000100
Epoch [14/20] - MSE Loss: 0.000047 | LR: 0.000100
Epoch [15/20] - MSE Loss: 0.000047 | LR: 0.000100
Epoch [16/20] - MSE Loss: 0.000047 | LR: 0.000100
Epoch [17/20] - MSE Loss: 0.000046 | LR: 0.000100
Epoch [18/20] - MSE Loss: 0.000046 | LR: 0.000100
Epoch [19/20] - MSE Loss: 0.000046 |

In [ ]:
torch.save(model_l2.state_dict(), "geolocation-prediction/saved_models/layer2_model_weights_v2.pth")
print("\nSuccess! Extra 25 epochs finished. Weights saved as _v2.pth")


Success! Extra 25 epochs finished. Weights saved as _v2.pth


In [25]:
# --- 1. Load Both Models ---
print("Loading Layer 1 and Layer 2 models...")
model_l1 = GeolocationMLP().to(device)
model_l1.load_state_dict(torch.load("geolocation-prediction/saved_models/layer1_model_weights.pth"))
model_l1.eval()
model_l2 = Layer2OffsetMLP().to(device)
model_l2.load_state_dict(torch.load("geolocation-prediction/saved_models/layer2_model_weights.pth"))
model_l2.eval()
centroids_3d_160_dev = centroids_3d_160.to(device)
def cartesian_to_latlon_tensor(x, y, z):
    lon_rad = torch.atan2(y, x)
    hyp = torch.sqrt(x * x + y * y)
    lat_rad = torch.atan2(z, hyp)
    return torch.rad2deg(lat_rad).cpu().numpy(), torch.rad2deg(lon_rad).cpu().numpy()

Loading Layer 1 and Layer 2 models...


In [26]:
# --- 2. PRECOMPUTE COSINE SIMILARITIES ---
print("Computing 115 Million Cosine Similarities...")
train_feats_norm = torch.nn.functional.normalize(features.to(device), p=2, dim=1)
test_feats_norm = torch.nn.functional.normalize(test_features.to(device), p=2, dim=1)
similarity_matrix = torch.matmul(test_feats_norm, train_feats_norm.T)
max_sims, best_train_indices = torch.max(similarity_matrix, dim=1)
max_sims = max_sims.cpu().numpy()
best_train_indices = best_train_indices.cpu().numpy()


Computing 115 Million Cosine Similarities...


In [92]:
# --- 3. Cascading Inference with k-NN Fallback ---
print("Running final inference...")
pred_lats, pred_lons, pred_radii = [], [], []
batch_size = 512
SIMILARITY_THRESHOLD = 0.96
with torch.no_grad():
    for i in range(0, len(test_features), batch_size):
        batch_feat = test_features[i:i+batch_size].to(device)
        
        # --- LAYER 1 ---
        out_16_logits, out_160_logits = model_l1(batch_feat)
        prob_16 = torch.softmax(out_16_logits, dim=1)
        prob_160 = torch.softmax(out_160_logits, dim=1)
        preds_16 = torch.argmax(out_16_logits, dim=1)
        preds_160 = torch.argmax(out_160_logits, dim=1)
        
        # --- LAYER 2 ---
        l2_inputs = torch.cat([batch_feat, prob_16, prob_160], dim=1)
        all_offsets = model_l2(l2_inputs)
        
        batch_indices = torch.arange(batch_feat.size(0))
        specific_offsets = all_offsets[batch_indices, preds_160, :]
        base_3d = centroids_3d_160_dev[preds_160]
        final_3d = base_3d + specific_offsets
        
        lats, lons = cartesian_to_latlon_tensor(final_3d[:, 0], final_3d[:, 1], final_3d[:, 2])
        
        # Iterate through the batch to apply fallback and custom radius logic
        for j in range(len(batch_feat)):
            global_idx = i + j
            base_radius = radius_map_16[int(preds_16[j].cpu())]
            
            if max_sims[global_idx] >= SIMILARITY_THRESHOLD:
                # k-NN OVERRIDE: Overwrite MLP coordinates with the exact training coordinates
                matched_train_idx = best_train_indices[global_idx]
                pred_lats.append(float(latitudes[matched_train_idx]))
                pred_lons.append(float(longitudes[matched_train_idx]))
                
                # Apply extra 0.4 multiplier for exact matches
                pred_radii.append(base_radius * 0.1 * 0.1)
            else:
                # MLP DEFAULT: Keep MLP predicted coordinates
                pred_lats.append(float(lats[j]))
                pred_lons.append(float(lons[j]))
                
                # Standard radius multiplier
                pred_radii.append(base_radius * 0.1)

Running final inference...


In [93]:
# --- 4. Save Final Submission ---
print(f"Bypassed MLP on {(max_sims >= SIMILARITY_THRESHOLD).sum()} out of {len(test_features)} images!")
print("Formatting Kaggle CSV...")
submission_df = pd.DataFrame({
    'image_id': test_image_ids,
    'pred_lat': pred_lats,
    'pred_lon': pred_lons,
    'pred_radius_km': pred_radii
})
submission_df['image_id'] = submission_df['image_id'].apply(lambda x: x + ".jpg" if not str(x).endswith('.jpg') else x)
submission_df.to_csv("geolocation-prediction/submissions/submission_knn_and_mlp.csv", index=False)
print("Done! Submission saved.")

Bypassed MLP on 274 out of 500 images!
Formatting Kaggle CSV...
Done! Submission saved.
